# Regressão Logística

In [183]:
import numpy as np

Problema: classificação de e-mails como spam

In [184]:
# links | palavra "grátis" | classificação (y)
# y = 1 -> spam
# y = 0 -> não spam
dados = np.array([[0, 0, 0],
                 [3, 1, 1],
                 [1, 0, 0],
                 [5, 1, 1]])

print(dados)

[[0 0 0]
 [3 1 1]
 [1 0 0]
 [5 1 1]]


$$ z = b_1 x_1 + b_2 x_2 + (...) + b_j x_j + b_0 = b^T x $$

- $ x_j $: valor da j-ésima característica
- $ b_j $: peso aprendido para a j-ésima característica ($w_j$ e $b_j$ são iguais)
- $ b_0 $: viés (intercepto)
- $ z \in \R $: pontuação que ainda não é probabilidade

Função logística (sigmoide)

$$ \sigma(z) = \frac{1}{1 + e^{-z}}$$

- Recebe $ z \in \R $
- Retorna $\sigma(z) \in (0, 1) $

In [185]:
def sigmoide(z):
    return 1 / (1 + np.e**(-z))

Exemplo:

$\sigma(2) \approx 0.881 $

Probabilidades: 88.1% para a classe 1, 11.9% para a classe 0.

Ajuste dos parâmetros -> Entropia cruzada binária

Não usamos o MSE porque os erros ficam extremamente pequenos (os erros podem ser altos, mas o MSE fica pequeno)

$$ L(p, y) = \begin{cases} -ln(p), & \text{se } y = 1, \\ -ln(1 - p), & \text{se } y = 0. \end{cases} $$

- para $y = 1$ -> queremos $p$ próximo de 1
- para $y = 0$ -> queremos $p$ próximo de 0


$$ L(p, y) = -[y ln(p) + (1-y) ln(1 - p)] $$

$$ J(w,b) = \frac{1}{m} \sum_{i=1}^{m} L(p^{(i)}, y^{(i)}) $$

Atualização dos pesos por descida do gradiente

$$ w_j \leftarrow w_j - \alpha \frac{\partial  J}{\partial  w_j} $$

$$ b \leftarrow b - \alpha \frac{\partial  J}{\partial  b} $$

$$ \frac{\partial  J}{\partial  w_j} = \frac{1}{m} \sum_{i=1}^{m} (p^{(i)} - y^{(i)}) x_j^{(i)}$$

$$ \frac{\partial  J}{\partial  b} = \frac{1}{m} \sum_{i=1}^{m} (p^{(i)} - y^{(i)})$$

No final das contas:

$$ \frac{1}{m} (p - y) x_j $$

para $b_0$, não há o termo '$x_j$'

In [186]:
def regressao_logistica(x, y, learning_rate = 0.1):

    m, n = x.shape

    pesos = np.zeros(n)
    
    b0 = 0

    epocas = 1000

    for epoca in range(epocas):
        z = np.dot(x, pesos) + b0
        
        p = sigmoide(z)

        # calculando gradientes de cada coeficiente e do intercepto
        dw = (1 / m) * np.dot(x.T, (p - y))

        db = (1 / m) * np.sum (p - y)

        pesos -= learning_rate * dw

        b0 -= learning_rate * db

    return b0, pesos

In [187]:
x = dados[:, :-1]
y = dados[:, -1]

b0, coefs = regressao_logistica(x, y)

print(f"Intecepto: {b0}")
print(f"Coeficientes: ")

for i in range (len(coefs)):
    print(f"x{i}: {coefs[i]}")

Intecepto: -4.419592072184113
Coeficientes: 
x0: 1.9422025056814036
x1: 2.0220224631832613


Limiar de decisão

$$ \hat{y} = \begin{cases} 1, & \text{se } \hat{p} \ge t , \\ 0, & \text{se } \hat{p} < t. \end{cases} $$

- Limiar menor: tende a detectar mais positivos, com mais falsos alarmes
- Limiar maior: tende a reduzir falsos alarmes, mas pode perder positivos

In [188]:
def limiar_de_decisao(p, t):
    if p >= t:
        return 1
    return 0

In [189]:
def classificacao(x, pesos, limiar):
    
    x_temp = np.ones(x.shape[0] + 1)
    
    for i in range(0, x.shape[0]):
        x_temp[i + 1] = x[i]

    coefs = pesos[1:]

    z = pesos[0] + np.dot(x, coefs)

    sig = sigmoide(z)

    classe = limiar_de_decisao(sig, 0.5)

    return classe

In [190]:
pesos = np.array([b0, coefs[0], coefs[1]])

print("Classe 0 -> não spam")
print("Classe 1 -> spam")

teste1 = classificacao(np.array([0, 1]), pesos, 0.5)
print(f"Teste 1: classe {teste1}")

teste2 = classificacao(np.array([4, 0]), pesos, 0.5)
print(f"Teste 2: classe {teste2}")

Classe 0 -> não spam
Classe 1 -> spam
Teste 1: classe 0
Teste 2: classe 1
